# PLM Novelty Retrieval
Summary of [ESM2](https://github.com/facebookresearch/esm)-based MIBiG novelty retrieval results from project: `[{{ project().name }}]`

## Description
> An additive retrieval side-channel: biosynthetic-core proteins from each antiSMASH region are embedded with a frozen ESM2 protein language model and queried against a [MIBiG](https://mibig.secondarymetabolites.org) reference index (built from antiSMASH's own KnownClusterBlast catalogue). Proteins whose nearest MIBiG neighbor is farther than a distance threshold are flagged as **novel** — i.e. divergent from anything currently cataloged in MIBiG.
>
> This is evidence, not a verdict: it never overwrites or replaces antiSMASH's own BGC calls or KnownClusterBlast hits. It surfaces an additional, auditable "how similar is this to known biosynthesis?" signal alongside them.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
import altair as alt

import warnings
warnings.filterwarnings('ignore')

report_dir = Path("../")

In [ ]:
region_table = report_dir / "plm_novelty/region_novelty_summary.tsv"
protein_table = report_dir / "plm_novelty/bgc_novelty_retrieval.tsv"

df_region = pd.read_csv(region_table, sep="\t")
df_protein = pd.read_csv(protein_table, sep="\t")

## Result Summary

In [ ]:
if len(df_region) == 0:
    display(Markdown("No BGC regions with biosynthetic-core proteins were available for novelty retrieval in this project."))
else:
    n_regions = len(df_region)
    n_flagged = int(df_region['flagged_novel'].sum())
    n_proteins = len(df_protein)
    n_novel_proteins = int(df_protein['novelty_flag'].sum())

    text = f"""Out of **{n_regions}** BGC regions scored, **{n_flagged}** ({n_flagged / n_regions:.1%}) are flagged as **novel** \
(majority of their biosynthetic-core proteins have no close MIBiG neighbor). Across **{n_proteins}** core proteins queried, \
**{n_novel_proteins}** ({n_novel_proteins / n_proteins:.1%}) individually lack a close MIBiG hit."""
    display(Markdown(text))

> Note: "novel" here means *retrieval-divergent from MIBiG*, not experimentally validated as a new natural product. Treat flagged regions as **candidates for follow-up**, not as confirmed novel BGCs.

In [ ]:
if len(df_region) > 0:
    source = df_region.copy()
    source['Status'] = source['flagged_novel'].map({True: 'Flagged novel', False: 'Similar to MIBiG'})

    chart = alt.Chart(source).mark_bar().encode(
        x=alt.X('region_novelty_score:Q', bin=alt.Bin(maxbins=20), axis=alt.Axis(title='Region Novelty Score', format='%')),
        y=alt.Y('count()', axis=alt.Axis(title='BGC Regions')),
        color=alt.Color('Status:N'),
        tooltip=['strain_id', 'region_id', 'n_core_proteins', 'frac_with_close_hit', 'region_novelty_score']
    ).interactive()

    display(chart)

## Region Novelty Summary
Per-region aggregate: fraction of core proteins with a close MIBiG hit, and the resulting novelty score/flag.

[Download Table]({{ project().file_server() }}/plm_novelty/region_novelty_summary.tsv){:target="_blank" .md-button}

In [ ]:
df_region

## Per-Protein Retrieval Detail
Nearest-neighbor MIBiG hits for every biosynthetic-core protein queried, with cosine distance and confidence band.

[Download Table]({{ project().file_server() }}/plm_novelty/bgc_novelty_retrieval.tsv){:target="_blank" .md-button}

In [ ]:
df_protein

## References
<font size="2">
{% for i in project().rule_used['plm-novelty']['references'] %}
- {{ i }}
{% endfor %}
</font>